# Notebook 23b - Bounded Climate Acquisition

C1.2 supplements, but does not overwrite, Notebook 23's literature-scaled baseline. It records one bounded CarbonPlan attempt and merges only persisted MACAv2-METDATA county values.

## Back-cast correction

The original C1 claim that its back-cast used an observed record was inaccurate: its comparison values were fallback climatology proxies. C1.5 replaces that reference check with PRISM 4 km gridded observations for 1981-2005. The inherited path restrictions prohibit editing `TERRA_build_log.md`; this correction is recorded here, in processed metadata, and in `MANUAL_FETCH.md`.

In [ ]:
# Cell 0 - mandatory notebook-name check
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
names = sorted(p.name for p in (ROOT / 'notebooks').glob('*.ipynb'))
print('23b_climate_acquisition.ipynb:', names.count('23b_climate_acquisition.ipynb'), 'matching notebook(s)')
print('\n'.join(names))

## Methods-ready scenario doctrine

This project maps RCP4.5 -> the ssp245 slot and RCP8.5 -> the ssp370 slot. This is a documented approximation, not an equivalence - RCP8.5 runs somewhat higher forcing than SSP370. Every value sourced this way carries method: cmip5_rcp_as_ssp_proxy and a note field citing the RCP->SSP substitution.

## CarbonPlan bounded-attempt result

The prior artifact contains only CMRA/Atlas/SNOTEL/USFS endpoint probes, not a CarbonPlan transfer error. C1.2 opened CarbonPlan DeepSD lazily with time chunks, subset the study-county union bbox before computation, and requested ssp245/2030. It stopped at Broomfield County (08014): no 0.25-degree grid-cell centroid fell within the county. No second CarbonPlan restructuring was attempted.

In [ ]:
import json
from collections import Counter
payload = json.loads((ROOT / 'data/processed/county_climate_projections.json').read_text())
records = payload['records']
required = {'value', 'scenario', 'epoch', 'percentile', 'source', 'method', 'confidence'}
complete = sum(all(r.get(k) not in (None, '') for k in required) for r in records)
print('CarbonPlan:', payload['carbonplan_attempt']['status'], payload['carbonplan_attempt']['stop_reason'])
print('Coverage:', payload['c12_coverage'])
print('Attribution:', len(records), complete)
print('Methods:', Counter(r['method'] for r in records))

In [ ]:
# C1 legacy back-cast and C1.3 MACA historical comparison.
validation = payload['historical_validation']
print('C1 legacy:', validation['note'])
print(validation['rows'])
c13 = payload['historical_validation_c13']
print('C1.3 MACA historical:', c13['maca_period'], c13['models_completed'])
for row in c13['comparison']:
    print(row['county'], row['metric'], 'observed=', row['observed_record'], 'legacy=', (row['legacy_p10'], row['legacy_p90']), 'MACA=', (row['maca_p10'], row['maca_p90']), row['status'])
print('C1.3 confidence downgrades:', c13['confidence_downgrades'])
c14 = payload['historical_validation_c14']
print('C1.4 root cause:', c14['root_cause'])
print('Longitude diagnostic:', c14['longitude'])
print('Variables/units:', c14['variables_and_units'])
print('1950 Baca annual calculation:', c14['annual_1950_baca_calculation'])
print('C1.4 action:', c14['future_projection_action'])
c15 = payload['historical_validation_c15']
print('C1.5 observed source:', c15['observed_source'], c15['observed_period'])
for row in c15['comparison']:
    print(row['county'], row['metric'], 'proxy=', row['c1_fallback_proxy'], 'MACA=', (row['maca_p10'], row['maca_p90']), 'PRISM=', row['prism_observed_value'], row['status'])
print('C1.5 confidence scope:', c15['confidence_scope'], c15['confidence_downgraded_records'])